# RetailPulse — Customer Segmentation

## 1. Imports

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from IPython.display import display

sys.path.append(os.path.abspath('..'))
from src.segmentation import (
    preprocess_rfm, elbow_analysis, silhouette_analysis, select_best_k,
    run_kmeans, find_eps, run_dbscan,
    evaluate_clusters, reduce_to_2d,
    cluster_profiles, assign_business_labels
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('ready')

## 2. Load RFM Data

In [ ]:
rfm = pd.read_csv('../data/processed/rfm_scores.csv')
print('shape:', rfm.shape)
display(rfm.head(5))

## 3. Preprocess

Log-transform Monetary (right-skewed) then StandardScale all 3 features.

In [ ]:
X_scaled, rfm_clean = preprocess_rfm(rfm)
print('X_scaled shape:', X_scaled.shape)
display(rfm_clean.head())

## 4. DBSCAN — Detect and Remove Outliers

Run DBSCAN first to identify noise points, then remove them before K-Means.

In [ ]:
auto_eps      = find_eps(X_scaled, min_samples=5)
dbscan_labels = run_dbscan(X_scaled, eps=auto_eps, min_samples=5)

outlier_mask       = dbscan_labels == -1
X_clean            = X_scaled[~outlier_mask]
rfm_clean_filtered = rfm_clean[~outlier_mask].reset_index(drop=True)

print(f'noise points: {outlier_mask.sum()}')
print(f'X_clean shape: {X_clean.shape}')

## 5. K-Means — Find Optimal k

Size-aware selection: rejects any k where a cluster has fewer than 1% of customers. `min_k=3` (default) enforces at least 3 segments.

In [ ]:
inertia = elbow_analysis(X_clean, max_k=10)
sil     = silhouette_analysis(X_clean, max_k=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(inertia.keys()), list(inertia.values()), marker='o', color='steelblue')
axes[0].set_title('Elbow Curve')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[1].plot(list(sil.keys()), list(sil.values()), marker='o', color='coral')
axes[1].set_title('Silhouette Scores')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.savefig('../reports/figures/kmeans_selection.png', dpi=150)
plt.show()

print('\nsize-aware k selection (min 1% per cluster, min_k=3):')
best_k = select_best_k(X_clean, max_k=10)

## 6. K-Means — Fit with Best k

In [ ]:
kmeans_labels, km_model = run_kmeans(X_clean, k=best_k)
print('cluster sizes:')
print(pd.Series(kmeans_labels).value_counts().sort_index().to_string())

## 7. Cluster Evaluation

K-Means uses `X_clean` (5840 rows). DBSCAN uses full `X_scaled` — noise (-1) filtered internally.

In [ ]:
km_scores = evaluate_clusters(X_clean,  kmeans_labels)
db_scores = evaluate_clusters(X_scaled, dbscan_labels)

scores_df = pd.DataFrame([km_scores, db_scores], index=['K-Means', 'DBSCAN'])
print('Cluster Evaluation:')
display(scores_df)

**How to read these scores:**

DBSCAN DB score is lower because it found only 2 dense clusters
and excluded 38 outliers before scoring. This does not mean DBSCAN
is a better segmentation model — it covers fewer customers.
K-Means is preferred here because it segments all 5840 non-outlier
customers into actionable business groups.

Calinski-Harabasz is higher for K-Means because CH rewards more clusters covering more data points.

## 8. Visualization — PCA Scatter Plots

In [ ]:
coords_2d = reduce_to_2d(X_clean, random_state=42)

# K-Means scatter
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(coords_2d[:, 0], coords_2d[:, 1], c=kmeans_labels, cmap='tab10', alpha=0.6, s=10)
ax.set_title(f'K-Means Clusters (k={best_k})', fontweight='bold')
ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
plt.colorbar(scatter, ax=ax)
plt.tight_layout()
plt.savefig('../reports/figures/kmeans_scatter.png', dpi=150)
plt.show()

# DBSCAN scatter — noise in black, plotted separately
coords_2d_full = reduce_to_2d(X_scaled, random_state=42)
mask_noise = dbscan_labels == -1
mask_valid = ~mask_noise
fig, ax = plt.subplots(figsize=(10, 7))
if mask_valid.any():
    ax.scatter(coords_2d_full[mask_valid, 0], coords_2d_full[mask_valid, 1], c=dbscan_labels[mask_valid], cmap='tab10', alpha=0.6, s=10)
if mask_noise.any():
    ax.scatter(coords_2d_full[mask_noise, 0], coords_2d_full[mask_noise, 1], c='black', alpha=0.5, s=10, label='noise')
ax.set_title('DBSCAN Clusters (black = noise)', fontweight='bold')
ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/dbscan_scatter.png', dpi=150)
plt.show()

## 9. Cluster Profiles

Mean R, F, M per cluster using original unscaled values from `rfm_clean_filtered`.

In [ ]:
profiles = cluster_profiles(rfm_clean_filtered, kmeans_labels)
print('K-Means cluster profiles:')
display(profiles)

In [ ]:
profiles.plot(kind='bar', figsize=(12, 5), colormap='Set2', edgecolor='white')
plt.title('Mean RFM per Cluster', fontweight='bold')
plt.xlabel('Cluster')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../reports/figures/cluster_profiles.png', dpi=150)
plt.show()

## 10. Business Labels

Labels assigned by relative ranking of cluster mean RFM — no hardcoded thresholds.
DBSCAN noise points (label = -1) are marked as Outlier.

In [ ]:
km_business_labels = assign_business_labels(rfm_clean_filtered, kmeans_labels)
print('K-Means business label distribution:')
print(km_business_labels.value_counts().to_string())

## 11. Re-attach Outliers and Save

In [ ]:
# clean rows
rfm_segment = rfm[~outlier_mask].copy().reset_index(drop=True)
rfm_segment['KMeans_Cluster'] = kmeans_labels
rfm_segment['DBSCAN_Cluster'] = dbscan_labels[~outlier_mask]
rfm_segment['Business_Label'] = km_business_labels.values

# outlier rows
rfm_outliers = rfm[outlier_mask].copy().reset_index(drop=True)
rfm_outliers['KMeans_Cluster'] = -1
rfm_outliers['DBSCAN_Cluster'] = -1
rfm_outliers['Business_Label'] = 'Outlier'

# merge and save
final = pd.concat([rfm_segment, rfm_outliers], ignore_index=True)
final.to_csv('../data/processed/customer_segments.csv', index=False)
print(f'saved customer_segments.csv - shape: {final.shape}')
print('\nBusiness_Label distribution:')
print(final['Business_Label'].value_counts().to_string())
display(final.head(8))

## Done!

- K-Means: size-aware k selection (min_k=3 default), all clusters > 1% of customers
- DBSCAN: auto eps via kneed, 38 noise points marked as Outlier
- Saved customer_segments.csv — shape (5878, 7)